In [2]:
# Write PySpark code that:
# 1. Reads the game_events.csv file into a DataFrame.
# 2. Converts the "event_time" column to a timestamp.
# 3. For each player, calculate the duration of each session (time between login and logout).
# 4. Calculate the total number of sessions per player.
# 5. Calculate the average session duration per player.
#6. Show the results, including player ID, total sessions, and average session duration.

In [3]:
from pyspark.sql import SparkSession

In [60]:
from pyspark.sql.functions import to_timestamp, col, lag, datediff, sum, when, last,unix_timestamp
from pyspark.sql.window import Window

In [4]:
spark = SparkSession.builder.appName("game").getOrCreate()

25/03/31 19:14:41 WARN Utils: Your hostname, Nikhils-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.0.90 instead (on interface en0)
25/03/31 19:14:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/31 19:14:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/03/31 19:14:42 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/03/31 19:14:42 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/03/31 19:14:42 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/03/31 19:14:42 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
25/03/31 19:14:58 WARN GarbageCollectionMetrics: To enable non-built-in garbage col

In [9]:
df = spark.read.csv("game_events.csv", header=True)

In [10]:
df.show()

+---------+-------------------+--------------+
|player_id|         event_time|    event_type|
+---------+-------------------+--------------+
|        1|2023-01-15 10:00:00|         login|
|        1|2023-01-15 10:05:00|   level_start|
|        1|2023-01-15 10:15:00|level_complete|
|        1|2023-01-15 11:00:00|        logout|
|        2|2023-01-15 10:30:00|         login|
|        2|2023-01-15 10:40:00|   level_start|
|        2|2023-01-15 11:30:00|        logout|
|        1|2023-01-15 12:00:00|         login|
|        1|2023-01-15 12:10:00|   level_start|
|        1|2023-01-15 12:45:00|        logout|
|        3|2023-01-15 13:00:00|         login|
|        3|2023-01-15 13:20:00|   level_start|
|        3|2023-01-15 14:00:00|level_complete|
|        3|2023-01-15 15:00:00|        logout|
+---------+-------------------+--------------+



In [17]:
df.printSchema()

root
 |-- player_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)



In [16]:
df = df.withColumn("event_time",to_timestamp(col("event_time")))

In [22]:
window_spec = Window.partitionBy("player_id").orderBy("event_time")

In [55]:
lag_df = df.withColumn("login_time", last(when(col("event_type")=="login", col("event_time")), ignorenulls=True).over(window_spec))

In [56]:
lag_df.show()

+---------+-------------------+--------------+-------------------+
|player_id|         event_time|    event_type|         login_time|
+---------+-------------------+--------------+-------------------+
|        1|2023-01-15 10:00:00|         login|2023-01-15 10:00:00|
|        1|2023-01-15 10:05:00|   level_start|2023-01-15 10:00:00|
|        1|2023-01-15 10:15:00|level_complete|2023-01-15 10:00:00|
|        1|2023-01-15 11:00:00|        logout|2023-01-15 10:00:00|
|        1|2023-01-15 12:00:00|         login|2023-01-15 12:00:00|
|        1|2023-01-15 12:10:00|   level_start|2023-01-15 12:00:00|
|        1|2023-01-15 12:45:00|        logout|2023-01-15 12:00:00|
|        2|2023-01-15 10:30:00|         login|2023-01-15 10:30:00|
|        2|2023-01-15 10:40:00|   level_start|2023-01-15 10:30:00|
|        2|2023-01-15 11:30:00|        logout|2023-01-15 10:30:00|
|        3|2023-01-15 13:00:00|         login|2023-01-15 13:00:00|
|        3|2023-01-15 13:20:00|   level_start|2023-01-15 13:00

In [62]:
duration_df = lag_df.filter(col("event_type") == "logout").withColumn(
    "duration_seconds",
    unix_timestamp(col("event_time")) - unix_timestamp(col("login_time")),
)

In [63]:
duration_df.show()

+---------+-------------------+----------+-------------------+----------------+
|player_id|         event_time|event_type|         login_time|duration_seconds|
+---------+-------------------+----------+-------------------+----------------+
|        1|2023-01-15 11:00:00|    logout|2023-01-15 10:00:00|            3600|
|        1|2023-01-15 12:45:00|    logout|2023-01-15 12:00:00|            2700|
|        2|2023-01-15 11:30:00|    logout|2023-01-15 10:30:00|            3600|
|        3|2023-01-15 15:00:00|    logout|2023-01-15 13:00:00|            7200|
+---------+-------------------+----------+-------------------+----------------+



In [64]:
result_df = duration_df.groupBy("player_id").agg(
    (sum("duration_seconds") / 60).alias("total_duration_minutes")
)

In [65]:
result_df.show()

+---------+----------------------+
|player_id|total_duration_minutes|
+---------+----------------------+
|        1|                 105.0|
|        2|                  60.0|
|        3|                 120.0|
+---------+----------------------+

